## Fact Purchases

Fato de vendas na camada Silver, integrada às dimensões analíticas.

In [0]:
# Imports
from pyspark.sql import functions as F

In [0]:
# Leitura das tabelas Silver 
df_purchase_header = spark.table("adventure_works_catalog.silver.clean_purchase_order")
df_purchase_detail = spark.table("adventure_works_catalog.silver.clean_purchase_order_line")

# Leitura das Dimensões
df_dim_product = spark.table("adventure_works_catalog.silver.dim_product")
df_dim_supplier = spark.table("adventure_works_catalog.silver.dim_supplier")
df_dim_date = spark.table("adventure_works_catalog.silver.dim_date")
df_dim_currency = spark.table("adventure_works_catalog.silver.dim_currency")


# Criação de coluna para comentários e documentação
def add_column_comments(catalog, schema, table, columns_dict):
    for column, comment in columns_dict.items():
        spark.sql(f"""
            ALTER TABLE `{catalog}`.`{schema}`.`{table}`
            ALTER COLUMN `{column}`
            COMMENT '{comment}'
        """)


In [0]:
# Join Header + Detail
df_purchase_base = (
    df_purchase_detail.alias("pd")
    .join(
        df_purchase_header.alias("ph"),
        on="PurchaseOrderID",
        how="inner"
    )
)


# Join com Dimensões 
df_purchase_enriched = (
    df_purchase_base

    # Product
    .join(
        df_dim_product.alias("p"),
        F.col("pd.ProductID") == F.col("p.ProductID"),
        "left"
    )

    # Supplier
    .join(
        df_dim_supplier.alias("s"),
        F.col("ph.VendorID") == F.col("s.SupplierID"),
        "left"
    )

    # Order Date
    .join(
        df_dim_date.alias("od"),
        F.to_date(F.col("ph.OrderDate")) == F.col("od.FullDate"),
        "left"
    )

    # Ship Date
    .join(
        df_dim_date.alias("sdte"),
        F.to_date(F.col("ph.ShipDate")) == F.col("sdte.FullDate"),
        "left"
    )

    # Currency (Purchasing usa moeda padrão, mas mantemos consistência)
    .join(
        df_dim_currency.alias("cur"),
        F.col("ph.OrderDate") == F.col("cur.CurrencyRateDate"),
        "left"
    )
)


# Seleção final
df_fact_purchases = (
    df_purchase_enriched
    .select(
        # Identificadores
        F.col("pd.PurchaseOrderDetailID").alias("PurchaseID"),
        F.col("pd.PurchaseOrderID").alias("OrderID"),

        # Chaves substitutas
        F.col("p.Product_SK").alias("Product_SK"),
        F.col("s.Supplier_SK").alias("Supplier_SK"),
        F.col("od.DateKey").alias("OrderDate_SK"),
        F.col("sdte.DateKey").alias("ShipDate_SK"),
        F.col("cur.Currency_SK").alias("Currency_SK"),

        # Métricas
        F.col("pd.OrderQuantity").alias("OrderQty"),
        F.col("pd.ReceivedQuantity"),
        F.col("pd.RejectedQuantity"),
        F.col("pd.StockedQuantity"),
        F.col("pd.UnitPrice"),
        F.col("pd.LineTotal"),

        # Custos do pedido
        F.col("ph.SubTotal"),
        F.col("ph.TaxAmount").alias("TaxAmt"),
        F.col("ph.Freight").alias("ShippingCost"),
        F.col("ph.TotalDue")
    )
)


# Garantia de Grão
df_fact_purchases = df_fact_purchases.dropDuplicates(
    ["PurchaseID", "OrderID"]
)


# Conferência
df_fact_purchases.printSchema()
df_fact_purchases.display()


# Escrita da Fato na Silver
(
    df_fact_purchases.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("adventure_works_catalog.silver.fact_purchases")
)

# Descrição das colunas
fact_purchases_columns = {
    "PurchaseID": "Identificador único do item da compra",
    "OrderID": "Identificador do pedido de compra",
    "Product_SK": "Chave substituta do produto",
    "Supplier_SK": "Chave substituta do fornecedor",
    "OrderDate_SK": "Chave da data do pedido de compra",
    "ShipDate_SK": "Chave da data de envio da compra",
    "Currency_SK": "Chave substituta da moeda",
    "OrderQty": "Quantidade solicitada na compra",
    "ReceivedQuantity": "Quantidade recebida do fornecedor",
    "RejectedQuantity": "Quantidade rejeitada na compra",
    "StockedQuantity": "Quantidade efetivamente estocada",
    "UnitPrice": "Preço unitário do produto na compra",
    "LineTotal": "Valor total do item comprado",
    "SubTotal": "Subtotal do pedido de compra",
    "TaxAmt": "Valor de imposto aplicado ao pedido de compra",
    "ShippingCost": "Custo de frete do pedido de compra",
    "TotalDue": "Valor total devido no pedido de compra"
}

# Adicionando comentários nas colunas
add_column_comments(
    catalog="adventure_works_catalog",
    schema="silver",
    table="fact_purchases",
    columns_dict=fact_purchases_columns
)

